# Beyond analytic solutions: tunnelling in a double-well potential

## A guided numerical exploration of the Schrödinger equation

In introductory quantum mechanics, we often choose potentials for which the Schrödinger equation can be solved analytically. Real systems are rarely so convenient. In this notebook, you will investigate a particle moving in a **double-well potential**, a simple model for molecular inversion, proton transfer, and other systems with two competing configurations.

You do **not** need to write or modify any code. Run each cell in order and use the interactive controls. The aim is to explore, make predictions, and explain what you observe.

### Learning goals

By the end of this notebook, you should be able to:

- explain how a differential equation becomes a matrix eigenvalue problem;
- identify tunnelling doublets and relate their energy splitting to a tunnelling time;
- distinguish an energy eigenstate from a localized, time-dependent state;
- explain how mass, barrier height, well separation, and asymmetry affect tunnelling;
- recognize an avoided crossing and connect it to state localization.


## 1. The physical model

We use the one-dimensional potential

$$
V(x)=V_0\left[\left(\frac{x}{a}\right)^2-1\right]^2+\epsilon\frac{x}{a}.
$$

For $\epsilon=0$, the two minima are at $x=\pm a$ and the central barrier has height $V_0$. The parameter $\epsilon$ tilts the potential, making one configuration energetically preferable.

The stationary Schrödinger equation is

$$
\left[-\frac{\hbar^2}{2m}\frac{d^2}{dx^2}+V(x)\right]\psi_n(x)=E_n\psi_n(x).
$$

Except for special parameter choices, this problem has no simple analytic solution. We place a grid over space and approximate the second derivative by

$$
\frac{d^2\psi}{dx^2}\bigg|_{x_i}\approx
\frac{\psi_{i+1}-2\psi_i+\psi_{i-1}}{(\Delta x)^2}.
$$

The differential equation then becomes a matrix eigenvalue problem, $H\boldsymbol{\psi}=E\boldsymbol{\psi}$. The computer diagonalizes $H$ to obtain the allowed energies and wavefunctions.

> This is a model, not a quantitative calculation for one specific molecule. The mass is an **effective mass** for the chosen coordinate.


In [ ]:
# Run this setup cell once
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown
from scipy.linalg import eigh_tridiagonal

plt.rcParams.update({"figure.figsize": (9, 5), "font.size": 11})

# Convenient units: energy in eV, distance in Å, mass in atomic mass units
HBAR2_OVER_2AMU = 0.00209008  # ħ²/(2 u), in eV Å²
HBAR_EV_FS = 0.6582119569      # ħ, in eV fs

def solve_double_well(V0=0.3, a=0.7, mass=0.5, tilt=0.0, n_states=10, N=900):
    """Return grid, potential, low energies, and normalized eigenfunctions."""
    L = max(4.5, 3.6*a)
    x = np.linspace(-L, L, N)
    dx = x[1] - x[0]
    V = V0*((x/a)**2 - 1.0)**2 + tilt*(x/a)
    kinetic = HBAR2_OVER_2AMU / mass
    diagonal = 2*kinetic/dx**2 + V
    off_diagonal = -kinetic/dx**2 * np.ones(N-1)
    E, psi = eigh_tridiagonal(diagonal, off_diagonal,
                              select='i', select_range=(0, n_states-1))
    # Convert Euclidean normalization into integral normalization
    psi = psi / np.sqrt(dx)
    return x, V, E, psi

def expectation_x(x, psi):
    dx = x[1] - x[0]
    return np.sum(np.abs(psi)**2 * x[:, None], axis=0) * dx

print("Setup complete — you can now run the interactive experiments.")

## 2. Stationary states and tunnelling doublets

Start with a symmetric potential ($\epsilon=0$). The two lowest wavefunctions are spread across both wells, even when the barrier is too high for a classical particle of the same energy to cross it. They form a closely spaced **tunnelling doublet**.

The curves below show $E_n+s\psi_n(x)$, so each wavefunction is displaced vertically to its energy. The vertical scale of $\psi_n$ is arbitrary; probabilities must be obtained from $|\psi_n|^2$.

Before moving the controls, predict what will happen to $E_1-E_0$ when the particle becomes heavier or the barrier becomes higher.


In [ ]:
def stationary_view(V0, a, mass, tilt, states):
    x, V, E, psi = solve_double_well(V0, a, mass, tilt, n_states=max(8, states))
    fig, ax = plt.subplots()
    ymax = min(max(V0*1.7, E[states-1]*1.25), 4.5)
    ax.plot(x, V, color='black', lw=2, label='Potential $V(x)$')
    colors = plt.cm.viridis(np.linspace(0.12, 0.88, states))
    scale = 0.09*ymax / max(np.max(np.abs(psi[:, :states])), 1e-12)
    for n in range(states):
        ax.axhline(E[n], color=colors[n], alpha=0.22, lw=0.8)
        ax.plot(x, E[n] + scale*psi[:, n], color=colors[n],
                label=f'$n={n}$,  $E={E[n]:.4f}$ eV')
    ax.axvline(0, color='0.7', ls=':', lw=1)
    ax.set(xlabel='Position $x$ (Å)', ylabel='Energy (eV)',
           xlim=(x[0], x[-1]), ylim=(min(-0.15, V.min()-0.1), ymax),
           title='Numerical eigenstates of the double well')
    ax.legend(fontsize=8, ncol=2, loc='upper center')
    ax.grid(alpha=0.18)
    plt.show()
    splitting = E[1]-E[0]
    half_period = np.pi*HBAR_EV_FS/splitting if splitting > 1e-12 else np.inf
    print(f"Ground-state doublet splitting: ΔE = {splitting:.6g} eV")
    print(f"Characteristic transfer time πħ/ΔE = {half_period:.4g} fs")

stationary_controls = {
    'V0': widgets.FloatSlider(value=0.3, min=0.1, max=2.0, step=0.1,
                              description='Barrier (eV):', continuous_update=False),
    'a': widgets.FloatSlider(value=0.7, min=0.55, max=1.6, step=0.05,
                             description='Half-distance (Å):', continuous_update=False),
    'mass': widgets.FloatLogSlider(value=0.5, base=10, min=-0.7, max=0.8, step=0.1,
                                   description='Mass (u):', continuous_update=False),
    'tilt': widgets.FloatSlider(value=0.0, min=-0.35, max=0.35, step=0.025,
                                description='Tilt (eV):', continuous_update=False),
    'states': widgets.IntSlider(value=4, min=2, max=6, step=1,
                                description='States:', continuous_update=False),
}
stationary_ui = widgets.VBox(list(stationary_controls.values()))
stationary_out = widgets.interactive_output(stationary_view, stationary_controls)
display(stationary_ui, stationary_out)

### Think about what you see

1. For a symmetric potential, why are the two lowest states not localized in separate wells?
2. Which change suppresses tunnelling more strongly in your trials: increasing the mass, increasing the barrier, or increasing the separation?
3. Find parameters for which the two lowest energy values appear almost identical. Are the wavefunctions identical?
4. Add a small tilt. Why can a small asymmetry have a large effect when the original splitting is tiny?
5. Can an eigenstate have an energy below the top of the barrier and still have nonzero amplitude on both sides?


## 3. Time-dependent Schrödinger equation: watch the particle tunnel

A stationary eigenstate changes only by a global phase, so its probability density does not move. To create a state localized in one well, we must form a **superposition** of eigenstates.

If the initial state is

$$
\Psi(x,0)=\sum_n c_n\psi_n(x),
$$

then

$$
\Psi(x,t)=\sum_n c_n\psi_n(x)e^{-iE_nt/\hbar}.
$$

The control below constructs the most left-localized combination of the lowest two eigenstates and evolves it exactly within that two-state subspace. Time is shown in units of $T_{\rm beat}=\pi\hbar/(E_1-E_0)$. For a symmetric well this is the transfer time; in an asymmetric well it remains the half-beat time, but transfer need not be complete.


In [ ]:
def dynamics_view(V0, a, mass, tilt, time_fraction):
    x, V, E, psi = solve_double_well(V0, a, mass, tilt, n_states=6)
    dx = x[1]-x[0]
    # Diagonalize x in the lowest two-state subspace: one combination is maximally left-localized
    X2 = (psi[:, :2].T @ (x[:, None]*psi[:, :2])) * dx
    loc_values, loc_vectors = np.linalg.eigh(X2)
    c = loc_vectors[:, np.argmin(loc_values)]
    delta = E[1]-E[0]
    beat_time = np.pi*HBAR_EV_FS/max(delta, 1e-15)
    t = time_fraction*beat_time
    phase = np.exp(-1j*E[:2]*t/HBAR_EV_FS)
    Psi = psi[:, :2] @ (c*phase)
    density = np.abs(Psi)**2
    P_left = np.sum(density[x < 0])*dx
    P_right = np.sum(density[x >= 0])*dx

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
    ax1.plot(x, V, 'k', lw=2)
    ax1.fill_between(x, 0, density/density.max()*max(0.25, V0*0.7),
                     color='#3b82f6', alpha=0.65, label='$|\\Psi(x,t)|^2$ (scaled)')
    ax1.axvline(0, color='0.65', ls=':')
    ax1.set(xlabel='$x$ (Å)', ylabel='Energy / scaled density',
            title=f'Wavepacket at $t/T_{{beat}}={time_fraction:.2f}$',
            xlim=(x[0], x[-1]), ylim=(min(-0.1, V.min()-0.05), min(3.0, max(V0*1.5, E[2]*1.2))))
    ax1.legend()
    ax1.grid(alpha=0.18)
    ax2.bar(['Left well', 'Right well'], [P_left, P_right],
            color=['#2563eb', '#f97316'])
    ax2.set_ylim(0, 1)
    ax2.set_ylabel('Probability')
    ax2.set_title(f'$P_L={P_left:.3f}$,  $P_R={P_right:.3f}$')
    ax2.grid(axis='y', alpha=0.2)
    plt.show()
    print(f"ΔE = {delta:.6g} eV; half-beat time πħ/ΔE = {beat_time:.4g} fs")

dynamics_controls = {
    'V0': widgets.FloatSlider(value=0.3, min=0.1, max=2.0, step=0.1,
                              description='Barrier (eV):', continuous_update=False),
    'a': widgets.FloatSlider(value=0.7, min=0.55, max=1.6, step=0.05,
                             description='Half-distance (Å):', continuous_update=False),
    'mass': widgets.FloatLogSlider(value=0.5, base=10, min=-0.7, max=0.8, step=0.1,
                                   description='Mass (u):', continuous_update=False),
    'tilt': widgets.FloatSlider(value=0.0, min=-0.25, max=0.25, step=0.025,
                                description='Tilt (eV):', continuous_update=False),
    'time_fraction': widgets.Play(value=0, min=0, max=200, step=2, interval=80,
                                  description='Play'),
}
# Use an integer proxy so the Play widget and slider remain synchronized in Colab
time_proxy = widgets.IntSlider(value=0, min=0, max=200, step=2, description='Time step:')
widgets.jslink((dynamics_controls['time_fraction'], 'value'), (time_proxy, 'value'))
def dynamics_from_proxy(V0, a, mass, tilt, time_step):
    dynamics_view(V0, a, mass, tilt, time_step/100)

dyn_args = {k:v for k,v in dynamics_controls.items() if k != 'time_fraction'}
dyn_args['time_step'] = time_proxy
dyn_ui = widgets.VBox([dynamics_controls['V0'], dynamics_controls['a'],
                       dynamics_controls['mass'], dynamics_controls['tilt'],
                       widgets.HBox([dynamics_controls['time_fraction'], time_proxy])])
dyn_out = widgets.interactive_output(dynamics_from_proxy, dyn_args)
display(dyn_ui, dyn_out)

### Questions for exploration

1. In the symmetric case, at what fractions of $T_{\rm transfer}$ is the particle mainly left, mainly right, or shared equally?
2. The particle crosses a region where $E<V(x)$. Does its total energy change during the process?
3. Increase the mass while keeping the potential fixed. Why does the motion slow dramatically?
4. Introduce a tilt. Does the particle still transfer completely to the other well? Explain this in terms of interference between the two energy eigenstates.
5. This model is relevant to isotope effects. What qualitative change would you expect when hydrogen is replaced by deuterium?


## 4. From tunnelling to localization: an avoided crossing

We now vary the tilt continuously. If the left and right wells were completely independent, their localized energy levels would cross when the wells exchanged order. Quantum tunnelling couples the two configurations: the exact eigenvalues repel one another and produce an **avoided crossing**.

The colour of each point below represents $\langle x\rangle$. Blue states are localized mainly on the left, orange states mainly on the right, and pale points are delocalized.


In [ ]:
def avoided_crossing_view(V0, a, mass, tilt_range):
    tilts = np.linspace(-tilt_range, tilt_range, 51)
    energies = np.zeros((len(tilts), 4))
    xpos = np.zeros_like(energies)
    for i, eps in enumerate(tilts):
        x, V, E, psi = solve_double_well(V0, a, mass, eps, n_states=4, N=650)
        energies[i] = E
        xpos[i] = expectation_x(x, psi)
    fig, ax = plt.subplots(figsize=(9, 5.5))
    for n in range(4):
        sc = ax.scatter(tilts, energies[:, n], c=xpos[:, n], cmap='coolwarm',
                        vmin=-a, vmax=a, s=24, label=f'$E_{n}$')
        ax.plot(tilts, energies[:, n], color='0.65', lw=0.6, zorder=0)
    cb = plt.colorbar(sc, ax=ax)
    cb.set_label('$\\langle x\\rangle$ (Å): left  ←  →  right')
    ax.axvline(0, color='black', ls=':', lw=1)
    ax.set(xlabel='Tilt $\\epsilon$ (eV)', ylabel='Energy (eV)',
           title='Energy levels and localization near an avoided crossing')
    ax.grid(alpha=0.18)
    ax.legend(ncol=4)
    plt.show()
    x0, V0arr, E0, psi0 = solve_double_well(V0, a, mass, 0, n_states=4, N=650)
    print(f"Minimum gap between the lowest two levels: {E0[1]-E0[0]:.6g} eV")

cross_controls = {
    'V0': widgets.FloatSlider(value=0.3, min=0.15, max=2.0, step=0.1,
                              description='Barrier (eV):', continuous_update=False),
    'a': widgets.FloatSlider(value=0.7, min=0.55, max=1.6, step=0.05,
                             description='Half-distance (Å):', continuous_update=False),
    'mass': widgets.FloatLogSlider(value=0.5, base=10, min=-0.7, max=0.8, step=0.1,
                                   description='Mass (u):', continuous_update=False),
    'tilt_range': widgets.FloatSlider(value=0.15, min=0.05, max=0.6, step=0.05,
                                      description='Tilt range:', continuous_update=False),
}
cross_ui = widgets.VBox(list(cross_controls.values()))
cross_out = widgets.interactive_output(avoided_crossing_view, cross_controls)
display(cross_ui, cross_out)

### Questions for exploration

1. Where is the gap between $E_0$ and $E_1$ smallest? How is it related to the tunnelling splitting in Section 2?
2. Far from zero tilt, where are the lowest two states localized?
3. Make the barrier higher or the particle heavier. What happens to the minimum gap?
4. In the limit of an extremely high and wide barrier, would you expect a true crossing or an avoided crossing? Why?
5. Imagine changing the tilt slowly from negative to positive. How might the system move from one well to the other without ever jumping over the barrier classically?


## 5. Challenge investigations

Choose one of the following investigations. There is no single required answer.

### A. Design a molecular switch

Find parameters for which the particle is localized by a modest tilt but tunnels rapidly when the potential is symmetric. Record your parameters and explain the compromise between localization and tunnelling speed.

### B. Investigate an isotope effect

Compare masses of $1\,\mathrm{u}$ and $2\,\mathrm{u}$ for the same potential. Calculate the ratio of their transfer times. Is the effect proportional to the mass ratio?

### C. Test the classical limit

Increase the mass and barrier separation. Describe how the eigenvalue spectrum, wavefunctions, and dynamics approach classical expectations. Which quantum feature survives longest?

### D. Break the symmetry

Find the smallest tilt for which the ground state has more than 90% probability in one well. Repeat for several barrier heights. Is the relevant comparison the tilt versus $V_0$, or the tilt versus the doublet splitting $\Delta E$?


## 6. What this numerical experiment has shown

- Discretizing space turns the Schrödinger differential equation into a matrix eigenvalue problem.
- In a symmetric double well, the energy eigenstates are delocalized and occur in tunnelling doublets.
- A localized state is a superposition; its time evolution comes from the relative phase accumulated by different energies.
- The tunnelling time is controlled by the splitting $\Delta E$ and is extremely sensitive to mass, barrier height, and separation.
- Asymmetry favours localization and can suppress complete transfer.
- Coupling between classically distinct configurations produces avoided crossings.

The double well is simple enough to visualize, but the same numerical ideas—constructing a Hamiltonian, finding eigenstates, and evolving superpositions—are used for atoms, molecules, nanostructures, and quantum devices whose Schrödinger equations cannot be solved analytically.


## 7. AI-assisted learning check — *Comprobación asistida por IAG*

This final activity uses generative AI as a **verification tool**, not as a substitute for your own reasoning.  
**Always answer first on your own.** Then use an AI assistant to identify possible errors, missing ideas or unclear reasoning, and compare its feedback with the notebook.

### 1. Write a ~200-word summary

In **about 200 words**, summarise what you have learned in this notebook. Your summary should include, in your own words:

- why the stationary eigenstates of a symmetric double well are delocalized;
- why a localized state requires a superposition of eigenstates;
- the meaning of the tunnelling splitting $\Delta E=E_1-E_0$;
- how $\Delta E$ is related to the transfer/tunnelling time;
- how mass, barrier height/separation and asymmetry affect tunnelling;
- what an avoided crossing tells us about coupled quantum states.

**Only after writing your summary**, paste it into a generative-AI tool and use the following prompt:

> *Act as a critical tutor in undergraduate quantum mechanics. Read my summary but do not rewrite it immediately. First: (1) identify any conceptual errors, (2) point out important ideas that are missing, and (3) identify any statements that are ambiguous or imprecise. Explain each issue briefly. Base your feedback on standard quantum mechanics and do not invent results that are not supported by my explanation.*

Revise your summary only after considering the feedback critically.

---

### 2. Self-explanation

Without using AI first, explain:

> **How can a particle initially localized in one well move to the other well even though the stationary energy eigenstates themselves do not move from one well to the other?**

Your explanation should explicitly mention **superposition**, **relative phase evolution**, and the role of the two lowest energy eigenstates.

Then ask the AI:

> *Check my explanation step by step. Do not simply give me the correct answer. Tell me which step of my reasoning is correct, incomplete or incorrect, and ask me one follow-up question that would help me improve the explanation.*

---

### 3. Two verification questions

Answer both questions **before** consulting the AI.

**Q1.** If the barrier is made higher or the particle mass is increased, what happens to the energy splitting $\Delta E=E_1-E_0$ and to the transfer time
$$
T_{\rm transfer}\approx\frac{\pi\hbar}{\Delta E}\,?
$$
Explain the physical reason for both changes.

**Q2.** In a perfectly symmetric double well the lowest eigenstates are delocalized. Why can introducing a small tilt localize the ground state and suppress complete transfer between the wells?

After answering, ask the AI:

> *Evaluate my answers to Q1 and Q2. For each one, state whether the reasoning is physically consistent and identify the first point that should be corrected, if any. Do not replace my answer with a model solution unless I explicitly ask for it.*

---

### 4. Final verification checklist

Before accepting the AI feedback, check:

- [ ] Does the explanation agree with the numerical results you obtained in this notebook?
- [ ] Does it conserve energy and respect the distinction between eigenstates and superpositions?
- [ ] Does it correctly relate tunnelling dynamics to $\Delta E$ rather than to a classical crossing of the barrier?
- [ ] If the AI disagrees with your result, can you verify the point using the equations, plots or simulations above?

> **Remember:** generative AI can produce plausible but incorrect explanations. The notebook results, the Schrödinger equation and your own physical reasoning remain the reference.
